In [466]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import plotly_express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR 
from sklearn.preprocessing import StandardScaler
import os
import webbrowser

In [467]:
# Loading data 
data = pd.read_csv('HistoricalQuotes.csv')
data.head()

,Date,Close/Last,Volume,Open,High,Low
0,02/28/2020,$273.36,106721200,$257.26,$278.41,$256.37
1,02/27/2020,$273.52,80151380,$281.1,$286,$272.96
2,02/26/2020,$292.65,49678430,$286.53,$297.88,$286.5
3,02/25/2020,$288.08,57668360,$300.95,$302.53,$286.13
4,02/24/2020,$298.18,55548830,$297.26,$304.18,$289.23


In [468]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2518 entries, 0 to 2517
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         2518 non-null   object
 1    Close/Last  2518 non-null   object
 2    Volume      2518 non-null   int64 
 3    Open        2518 non-null   object
 4    High        2518 non-null   object
 5    Low         2518 non-null   object
dtypes: int64(1), object(5)
memory usage: 118.2+ KB


In [469]:
data.columns

Index(['Date', ' Close/Last', ' Volume', ' Open', ' High', ' Low'], dtype='object')

In [470]:
# cleaning the name columns name 
data.columns = data.columns.str.strip()
data.rename(columns={'Close/Last' : 'Close'}, inplace=True)

In [471]:
data.columns

Index(['Date', 'Close', 'Volume', 'Open', 'High', 'Low'], dtype='object')

In [472]:
data.isnull().sum()

Date      0
Close     0
Volume    0
Open      0
High      0
Low       0
dtype: int64

In [473]:
data.duplicated().sum()

np.int64(0)

In [474]:
#cleaning the rows, removing $ sign
data['Close'] = data['Close'].str.replace('$', '').astype(float)
data['Open'] = data['Open'].str.replace('$', '').astype(float)
data['High'] = data['High'].str.replace('$', '').astype(float)
data['Low'] = data['Low'].str.replace('$', '').astype(float)
data['Volume'] = data['Volume'].astype(float)

In [475]:
data['Date'] = pd.to_datetime(data['Date'])
data.sort_values('Date',inplace=True)

In [476]:
# Creating target
data['Target'] = data['Close'].shift(-1)
data.dropna(inplace=True)

In [477]:
data.head()

,Date,Close,Volume,Open,High,Low,Target
2517,2010-03-01,29.8557,137312041.0,29.3928,29.9286,29.3500,29.8357
2516,2010-03-02,29.8357,141486282.0,29.9900,30.1186,29.6771,29.9043
2515,2010-03-03,29.9043,92846488.0,29.8486,29.9814,29.7057,30.1014
2514,2010-03-04,30.1014,89591907.0,29.8971,30.1314,29.8043,31.2786
2513,2010-03-05,31.2786,224647427.0,30.7057,31.3857,30.6614,31.2971


In [478]:
data['Close_lag1'] = data['Close'].shift(1)
data['Close_lag2'] = data['Close'].shift(2)
data['MA10'] = data['Close'].rolling(10).mean()
data['MA20'] = data['Close'].rolling(20).mean()
data.dropna(inplace=True)


In [479]:
data['Return'] = data['Close'].pct_change()
data['Volatility'] = data['Return'].rolling(10).std()
data['Momentum'] = data['Close'] - data['Close_lag1']
data['MA50'] = data['Close'].rolling(50).mean()
data['Return'] = data['Close'].pct_change()
data['Volatility'] = data['Return'].rolling(10).std()
data.dropna(inplace=True)

In [480]:
data.head()

,Date,Close,Volume,Open,High,Low,Target,Close_lag1,Close_lag2,MA10,MA20,Return,Volatility,Momentum,MA50
2449,2010-06-07,35.8486,221253336.0,36.8986,37.0214,35.7928,35.6186,36.5664,37.5886,36.30175,36.149295,-0.019630,0.019442,-0.7178,35.806512
2448,2010-06-08,35.6186,249904415.0,36.1771,36.2571,35.0928,34.7428,35.8486,36.5664,36.33847,36.116010,-0.006416,0.018938,-0.2300,35.859170
2447,2010-06-09,34.7428,213040094.0,35.9243,35.9857,34.6414,35.7871,35.6186,35.8486,36.30961,36.020865,-0.024588,0.020558,-0.8758,35.890054
2446,2010-06-10,35.7871,193507918.0,34.9771,35.8543,34.6000,36.2157,34.7428,35.6186,36.40105,35.938150,0.030058,0.022635,1.0443,35.931954
2445,2010-06-11,36.2157,136154451.0,35.4607,36.2657,35.3386,36.3257,35.7871,34.7428,36.40334,35.903505,0.011976,0.019438,0.4286,35.984840


In [481]:
# Feature traget 
X = data[['Close','Close_lag1','Close_lag2','MA10','MA20','Volume']]
Y = data['Target']

In [482]:
# Train Test Split 
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [483]:
model_lr = LinearRegression()
model_lr.fit(X_train,Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [484]:
y_pred = model_lr.predict(X_test)

In [485]:
mae = mean_absolute_error(Y_test,y_pred)
rmse = np.sqrt(mean_squared_error(Y_test,y_pred))
r2_score = r2_score(Y_test,y_pred)
print('MAE :',mae)
print('RMSE :',rmse)
print('R2 :',r2_score)

MAE : 1.3152541141761813
RMSE : 1.9343479117552766
R2 : 0.9988068756246817


In [486]:
#Prediction 
result = pd.DataFrame({
    'Actual target' : Y_test,
    'Predicted target' : y_pred
})
result.head(10)

,Actual target,Predicted target
1159,125.1600,125.426615
1463,83.9985,84.969365
84,243.2900,249.394384
1807,74.3097,74.114434
1842,82.4000,85.184173
990,105.6700,106.314766
974,107.4800,110.013249
418,187.1800,184.937737
590,157.1000,156.237491
72,262.6400,265.011718


In [487]:
((y_pred > X_test['Close']) == (Y_test > X_test['Close'])).mean()

np.float64(0.5408163265306123)

In [488]:
model_rf = RandomForestRegressor()
model_rf.fit(X_train,Y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [489]:
y_pred = model_rf.predict(X_test)

In [490]:
mae = mean_absolute_error(Y_test,y_pred)
rmse = np.sqrt(mean_squared_error(Y_test,y_pred))

print('RMSE :',rmse)
print('MAe :',mae)

RMSE : 2.1790493068149304
MAe : 1.448662942857144


In [491]:
#prediction
result = pd.DataFrame({
    'Actual target' : Y_test,
    'Predicted target ' : y_pred
})
result.head(10)

,Actual target,Predicted target
1159,125.1600,126.364700
1463,83.9985,84.625077
84,243.2900,247.639400
1807,74.3097,74.315951
1842,82.4000,85.658645
990,105.6700,106.356700
974,107.4800,110.279750
418,187.1800,185.566200
590,157.1000,156.898900
72,262.6400,264.875600


In [492]:
((y_pred > X_test['Close']) == (Y_test > X_test['Close'])).mean()

np.float64(0.5102040816326531)

In [493]:
model_xgb = XGBRegressor()
model_xgb.fit(X_train,Y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [494]:
y_pred = model_xgb.predict(X_test)

In [495]:
mae = mean_absolute_error(Y_test,y_pred)
rmse = np.sqrt(mean_squared_error(Y_test,y_pred))
print('MAE :', mae)
print('RMSe :', rmse)


MAE : 1.637300994686204
RMSe : 2.4768779042682505


In [496]:
result = pd.DataFrame({
    'Actual target' : Y_test,
    'Predicted target' : y_pred
})
result.head(10)

,Actual target,Predicted target
1159,125.1600,125.461349
1463,83.9985,84.641327
84,243.2900,245.670822
1807,74.3097,72.820747
1842,82.4000,85.405518
990,105.6700,107.420715
974,107.4800,109.739212
418,187.1800,183.500900
590,157.1000,155.179779
72,262.6400,263.336060


In [497]:
((y_pred > X_test['Close']) == (Y_test > X_test['Close'])).mean()

np.float64(0.5020408163265306)

In [498]:

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [499]:
model_svr = SVR()
model_svr.fit(X_train,Y_train)


,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [500]:
y_pred = model_svr.predict(X_test)

In [501]:
mae = mean_absolute_error(Y_test,y_pred)
rmse = np.sqrt(mean_squared_error(Y_test,y_pred))
print('MAE :', mae)
print('RMSE :',rmse)

MAE : 4.225613485808466
RMSE : 12.339230719412614


In [502]:
result = pd.DataFrame({
    'Actual target': Y_test,
    'Predicted target' : y_pred
})
result.head(10)

,Actual target,Predicted target
1159,125.1600,126.068516
1463,83.9985,84.179986
84,243.2900,227.451756
1807,74.3097,75.141227
1842,82.4000,87.172063
990,105.6700,106.338363
974,107.4800,111.299700
418,187.1800,187.036470
590,157.1000,156.734158
72,262.6400,227.128028


In [503]:
html_file_path="./dashboard/"
if not os.path.exists(html_file_path):
    os.makedirs(html_file_path)

plot_containers=""

In [504]:
def save_plot_as_html(fig,filename,insight):
    global plot_containers
    filepath = os.path.join(html_file_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    plot_containers += f"""
 <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath,full_html=False, include_plotlyjs='inline')


In [505]:
plot_width=500
plot_hight = 400
plot_bg_color = 'black'
text_color = 'white'
axis_font = {'size' : 16}
title_font = {"size" : 12}

In [506]:
#Price Trend
fig_price_plot = px.line(
    x= data['Date'],
    y = data['Close'],
    labels={'x' : 'Date','y' : 'Close Price'},
    title = 'Stock Price Over Time',
    color_discrete_sequence=px.colors.sequential.Plasma,
)
fig_price_plot.update_layout(
    font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig_price_plot.show()
fig_price_plot.update_traces(marker=dict(line=dict(color='white',width=1)))
insight_text = """
<b>Stock Price Trend Analysis:</b><br>
• Shows Apple stock price movement over time<br>
• Plasma color gradient indicates price intensity<br>
• Annotations mark the highest and lowest price points<br>
• Helps identify long-term price trends and volatility periods
"""

save_plot_as_html(fig_price_plot,"Price_Trend", insight_text)

In [507]:
#trend analysis
fig_moving_avg = px.line(
    data_frame=data,
    x='Date',
    y=['Close','MA10','MA20'],
    labels={'x': 'Date', 'y': 'Price', 'variable': 'Price Type'},
    title='Moving Averages',
    color_discrete_sequence=['blue','orange','green'],
)
fig_moving_avg.update_layout(
     font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig_moving_avg.show()
insight_text = """
<b>Moving Averages Analysis:</b><br>
• Blue line: Closing price<br>
• Orange line: 10-day moving average<br>
• Green line: 20-day moving average<br>
• Moving averages smooth out price data to identify trends<br>
• Shorter MAs react faster to price changes
"""
save_plot_as_html(fig_moving_avg,'Moving_averages',insight_text)


In [508]:
# Create a dataframe with actual vs predicted
pred_comparison = pd.DataFrame({
    'Index': range(len(Y_test)),
    'Actual': Y_test.values,
    'Predicted': y_pred
})

fig_pred = px.line(
    pred_comparison,
    x='Index',
    y=['Actual', 'Predicted'],
    labels={'x': 'Sample Index', 'y': 'Price', 'variable': 'Type'},
    title='Actual vs Predicted Stock Price',
    color_discrete_sequence=['blue', 'green']
)

fig_pred.update_layout(
    font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)

fig_pred.show()

insight_text = """
<b>Model Prediction Analysis:</b><br>
• Blue line: Actual test values<br>
• Orange line: Model predicted values<br>
• Closer alignment indicates better model performance<br>
• Divergences show where the model struggles with price prediction
"""

save_plot_as_html(fig_pred, 'Actual_vs_Predicted', insight_text)

In [509]:
# Calculate error
error = Y_test - y_pred

error_df = pd.DataFrame({'Error': error})

fig_error = px.histogram(
    error_df,
    x='Error',
    nbins=50,
    title='Error Distribution',
    color_discrete_sequence=['blue']
)

fig_error.update_layout(
    font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)

fig_error.show()

insight_text = """
<b>Error Distribution Analysis:</b><br>
• Shows the frequency distribution of prediction errors<br>
• Centered near zero indicates good model accuracy<br>
• Wide distribution suggests model struggles consistently<br>
• Symmetric distribution indicates balanced over/under predictions
"""

save_plot_as_html(fig_error, 'Error_Distribution', insight_text)

In [510]:
importances = model_rf.feature_importances_
features = X.columns
features_df = pd.DataFrame({
    'importances': importances,
    'features': features
})
fig_features = px.bar(
    features_df,
    x='features',
    y='importances',
    title='Feature Importances',
    color_discrete_sequence=['blue']
)
fig_features.update_layout(
    font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig_features.show()
insight_text = """
<b>Feature Importance Analysis:</b><br>
• Shows the relative importance of each feature in the Random Forest model<br>
• Higher bars indicate features that contribute more to predictions<br>
• Helps in understanding which variables drive the model's decisions
"""
save_plot_as_html(fig_features, 'Feature_Importance', insight_text)

In [511]:
importances = model_xgb.feature_importances_
features = X.columns

feature_df = pd.DataFrame({
    'importances' : importances,
    'features' : features
})
fig_features_XGB = px.bar(
    feature_df,
    x='features',
    y='importances',
    title='Feature Importances',
    color_discrete_sequence=['blue']
)
fig_features_XGB.update_layout(
     font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig_features_XGB.show()
insight_text = """
<b>Feature Importance Analysis:</b><br>
• Shows the relative importance of each feature in the Random Forest model<br>
• Higher bars indicate features that contribute more to predictions<br>
• Helps in understanding which variables drive the model's decisions
"""
save_plot_as_html(fig_features_XGB, 'Feature_Importance', insight_text)

In [512]:
# Correlation Matrix
corr_matrix = data.corr()

fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    title="Correlation Matrix",
    color_continuous_scale='RdBu_r'  
)

fig_corr.update_layout(
    font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)

fig_corr.show()

insight_text = """
<b>Correlation Matrix Analysis:</b><br>
• Shows relationships between numerical variables<br>
• Values range from -1 (perfect negative correlation) to +1 (perfect positive correlation)<br>
• 0 indicates no linear relationship<br>
• Red indicates positive correlation, blue indicates negative correlation<br>
• Helps identify multicollinearity and feature relationships
"""

save_plot_as_html(fig_corr, 'Correlation_Matrix', insight_text)

In [513]:
fig_return = px.line(
    x=data['Date'],
    y=data['Return'],
    labels={'x':'Date','y':'Return'},
    title='Daily Returns',
)
fig_return.update_layout(
     font=dict(color='black'),
    title=dict(font=title_font),
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig_return.show()
insight_text = """
<b>Daily Returns Analysis:</b><br>
• Shows percentage change in stock price each day<br>
• Positive values indicate price increases, negative indicate decreases<br>
• Annotations highlight the best and worst daily performance<br>
• Helps assess daily volatility and market reactions
"""

save_plot_as_html(fig_return, 'Daily_Returns', insight_text)

In [514]:
# plot_containers_split = plot_containers.split('</div>')

In [515]:
# if len(plot_containers_split) > 1:
#     final_plot=plot_containers_split[-2]+'</div>'
# else:
#     final_plot=plot_containers

In [516]:
dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">

<title>Apple Stock AI Dashboard</title>

<style>

/* RESET */
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

/* BACKGROUND WITH STOCK FEEL */
body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto;
    color: white;
    min-height: 100vh;
    overflow-x: hidden;
    
    background: linear-gradient(120deg, #020617, #0f172a);
}

/* ANIMATED STOCK LINE BACKGROUND */
body::before {
    content: "";
    position: fixed;
    width: 200%;
    height: 200%;
    background: repeating-linear-gradient(
        120deg,
        rgba(59,130,246,0.08) 0px,
        rgba(59,130,246,0.08) 2px,
        transparent 2px,
        transparent 40px
    );
    animation: moveBg 20s linear infinite;
    z-index: 0;
}

@keyframes moveBg {
    0% { transform: translate(0,0); }
    100% { transform: translate(-200px, -200px); }
}

/* HEADER */
.header {
    text-align: center;
    padding: 30px 20px;
    position: relative;
    z-index: 1;
}

.header h1 {
    font-size: 38px;
    font-weight: 600;
    background: linear-gradient(90deg, #60a5fa, #22d3ee);
    -webkit-background-clip: text;
    color: transparent;
}

.header p {
    color: #94a3b8;
    margin-top: 8px;
}

/* CONTAINER - VERTICAL */
.container {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 30px;
    padding: 30px;
    position: relative;
    z-index: 1;
}

/* CARD */
.card {
    width: 90%;
    max-width: 950px;
    background: rgba(255,255,255,0.05);
    border-radius: 20px;
    padding: 15px;
    backdrop-filter: blur(25px);
    border: 1px solid rgba(255,255,255,0.1);
    transition: 0.4s;
    position: relative;
    overflow: hidden;

    animation: fadeUp 0.6s ease;
}

.card:hover {
    transform: translateY(-10px) scale(1.01);
    box-shadow: 0 25px 50px rgba(0,0,0,0.6);
}

/* IMAGE */
.card img {
    width: 100%;
    height: 280px;
    object-fit: cover;
    border-radius: 14px;
    transition: 0.3s;
}

.card:hover img {
    transform: scale(1.04);
}

/* TITLE */
.card h3 {
    margin-top: 12px;
    font-size: 20px;
    color: #e2e8f0;
}

/* BUTTON */
.btn {
    margin-top: 12px;
    padding: 10px 16px;
    background: linear-gradient(90deg, #3b82f6, #06b6d4);
    border: none;
    border-radius: 12px;
    color: white;
    cursor: pointer;
    transition: 0.3s;
}

.btn:hover {
    transform: scale(1.07);
    box-shadow: 0 10px 25px rgba(59,130,246,0.6);
}

/* INSIGHT */
.insight {
    position: absolute;
    bottom: 12px;
    left: 12px;
    right: 12px;
    background: rgba(0,0,0,0.75);
    padding: 10px;
    border-radius: 10px;
    font-size: 13px;
    opacity: 0;
    transition: 0.3s;
}

.card:hover .insight {
    opacity: 1;
}

/* ENTRY ANIMATION */
@keyframes fadeUp {
    from {
        opacity: 0;
        transform: translateY(40px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}

</style>

<script>
function openPlot(file) {
    window.open(file, '_blank');
}
</script>

</head>

<body>

<div class="header">
    <h1> Apple Stock Market Prediction</h1>
    <p>AI Powered Analysis • Smart Forecast • Clean Visualization</p>
</div>

<div class="container">
{plots}
</div>

</body>
</html>
"""

In [517]:
final_html = dashboard_html.replace("{plots}", plot_containers)

In [518]:
dashboard_path=os.path.join(html_file_path,"index.html")

In [519]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

In [520]:
webbrowser.open('file://'+os.path.realpath(dashboard_path))

True